# Inference and Submission
Final notebook for loading checkpoints and writing submission.csv.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchaudio
from tqdm.auto import tqdm

sys.path.insert(0, "/kaggle/working/birdclef2026")

from src.train import load_config
from src.model import build_model
from src.transforms import build_mel_transform
from src.metrics import topn_postprocessing
from src.temporal import extract_hour_from_window, load_temporal_prior, apply_temporal_prior

CONFIG_PATH = "/kaggle/working/birdclef2026/configs/kaggle_base.yaml"
config = load_config(CONFIG_PATH)
config["config_path"] = CONFIG_PATH

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SAMPLE_SUB_PATH = f"{config['paths']['base']}/sample_submission.csv"
CHECKPOINT_PATH = "/kaggle/working/checkpoints/best_checkpoint.pth"
if not Path(CHECKPOINT_PATH).exists():
    CHECKPOINT_PATH = "/kaggle/working/checkpoints/kaggle_model_state.pth"

USE_TOPN = True
TOPN_N = 1
USE_TEMPORAL_PRIOR = False

# Optional debugging
MAX_FILES = None  # e.g. 2 for quick test
BATCH_SIZE = config.get("inference", {}).get("batch_size", 64)


In [ ]:
# Build model + mel transform
model_config = config.get("model", {})
model = build_model(
    num_classes=config["project"]["num_classes"],
    backbone=model_config.get("backbone", "efficientnet_b0"),
    pretrained=model_config.get("pretrained", True),
    gem_p=model_config.get("gem_p", 3.0),
    head_dropout=model_config.get("head_dropout", 0.25),
    use_time_conditioning=False,
    in_channels=1,
).to(DEVICE)

mel_transform = build_mel_transform(config).to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model.load_state_dict(state, strict=True)
model.eval()

print("Loaded checkpoint:", CHECKPOINT_PATH)


In [ ]:
# Helper functions
sample_rate = config["audio"]["sample_rate"]
chunk_seconds = config["audio"]["chunk_duration"]
chunk_samples = int(sample_rate * chunk_seconds)


def load_audio(path: Path) -> torch.Tensor:
    wav, sr = torchaudio.load(path)
    if wav.ndim == 2:
        wav = wav.mean(dim=0)
    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, sr, sample_rate)
    return wav


def get_window(wav: torch.Tensor, start_sec: int) -> torch.Tensor:
    if len(wav) < chunk_samples:
        n_repeats = (chunk_samples // len(wav)) + 1
        wav = wav.repeat(n_repeats)[:chunk_samples]
        return wav

    start = int(start_sec * sample_rate)
    end = start + chunk_samples
    if end > len(wav):
        start = max(0, len(wav) - chunk_samples)
        end = start + chunk_samples
    return wav[start:end]


def batch_predict(windows: list[torch.Tensor]) -> np.ndarray:
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(windows), BATCH_SIZE):
            batch = torch.stack(windows[i : i + BATCH_SIZE]).to(DEVICE)
            mel = mel_transform(batch)
            mel = torch.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
            _, probs = model(mel)
            all_preds.append(probs.detach().cpu().numpy())
    return np.concatenate(all_preds, axis=0)


def parse_row_id(row_id: str) -> tuple[str, int]:
    stem, offset = row_id.rsplit("_", 1)
    return f"{stem}.ogg", int(offset)


In [ ]:
# Inference + submission
sample_df = pd.read_csv(SAMPLE_SUB_PATH)
species_cols = sample_df.columns[1:].tolist()

# Map model output order (taxonomy.csv) to submission column order
taxonomy_df = pd.read_csv(config["paths"]["taxonomy"])
primary_labels = taxonomy_df["primary_label"].astype(str).tolist()
index_map = []
for sp in species_cols:
    if sp not in primary_labels:
        raise ValueError(f"Species {sp} missing from taxonomy.csv")
    index_map.append(primary_labels.index(sp))

rowid_to_idx = {rid: i for i, rid in enumerate(sample_df["row_id"].tolist())}

# Group rows by file
file_to_offsets = {}
for rid in sample_df["row_id"].tolist():
    fname, offset = parse_row_id(rid)
    file_to_offsets.setdefault(fname, []).append(offset)

files = sorted(file_to_offsets.keys())
if MAX_FILES is not None:
    files = files[:MAX_FILES]

use_temporal = USE_TEMPORAL_PRIOR
if use_temporal:
    prior_species, prior_class = load_temporal_prior(config["paths"]["precomputed"])
else:
    prior_species, prior_class = None, None

pred_matrix = np.zeros((len(sample_df), len(species_cols)), dtype=np.float32)

for fname in tqdm(files, desc="Files"):
    path = Path(config["paths"]["test_soundscapes"]) / fname
    if not path.exists():
        raise FileNotFoundError(f"Missing soundscape file: {path}")

    wav = load_audio(path)
    offsets = sorted(file_to_offsets[fname])
    windows = [get_window(wav, s) for s in offsets]

    preds = batch_predict(windows)
    preds = preds[:, index_map]
    preds = np.nan_to_num(preds, nan=0.0, posinf=1.0, neginf=0.0)

    if USE_TOPN:
        preds = topn_postprocessing({fname: preds}, n=TOPN_N)[fname]

    if use_temporal:
        for i, sec in enumerate(offsets):
            hour = extract_hour_from_window(fname, window_start_seconds=float(sec))
            preds[i] = apply_temporal_prior(
                preds[i],
                hour=hour,
                prior_species=prior_species,
                prior_class=prior_class,
                taxonomy_df=taxonomy_df,
                use_class_fallback=True,
            )

    for i, sec in enumerate(offsets):
        row_id = f"{Path(fname).stem}_{sec}"
        pred_matrix[rowid_to_idx[row_id]] = preds[i]

submission = pd.DataFrame(pred_matrix, columns=species_cols)
submission.insert(0, "row_id", sample_df["row_id"].tolist())
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
